# 04: Black-Box Baselines (Stage 3)

Fine-tune EfficientNet-B0 and ResNet-50 on HAM10000. Saves Grad-CAM-compatible checkpoints for later faithfulness comparison (Stage 6).

In [1]:
import os, sys, yaml, logging
from pathlib import Path
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

import torch, numpy as np
from torch.utils.data import DataLoader, Subset

from src.data.datasets import build_dataset, default_collate
from src.data.transforms import baseline_train_transform, baseline_eval_transform
from src.models.baseline_cnn import (
    build_baseline, train_baseline, evaluate_baseline, predict_baseline,
)
from src.utils import get_device, describe_device, dataloader_kwargs, seed_everything

logging.basicConfig(level=logging.INFO)
cfg = yaml.safe_load((PROJECT_ROOT / 'config.yaml').read_text())
seed_everything(cfg.get('seed', 42))
device = get_device()
print('device:', describe_device(device))

device: mps (Apple arm64)


In [2]:
# Build HAM10000 loaders (80/10/10 split)
root = PROJECT_ROOT / cfg['paths']['datasets']['ham10000']
train_ds = build_dataset('ham10000', root=root,
                         transform=baseline_train_transform(cfg['baseline']['image_size']))
eval_ds = build_dataset('ham10000', root=root,
                        transform=baseline_eval_transform(cfg['baseline']['image_size']))
n = len(train_ds)
rng = np.random.default_rng(cfg['seed'])
idx = np.arange(n)
rng.shuffle(idx)
tr = idx[:int(0.8 * n)]; va = idx[int(0.8 * n):int(0.9 * n)]; te = idx[int(0.9 * n):]

loader_kwargs = dataloader_kwargs(device=device)
batch = cfg['baseline']['batch_size']
train_loader = DataLoader(Subset(train_ds, tr), batch_size=batch, shuffle=True,
                          collate_fn=default_collate, **loader_kwargs)
val_loader   = DataLoader(Subset(eval_ds,  va), batch_size=batch, shuffle=False,
                          collate_fn=default_collate, **loader_kwargs)
test_loader  = DataLoader(Subset(eval_ds,  te), batch_size=batch, shuffle=False,
                          collate_fn=default_collate, **loader_kwargs)
print(f'train={len(tr)} val={len(va)} test={len(te)}  batch={batch}')

train=8012 val=1001 test=1002  batch=16


In [3]:
# Class weights (HAM10000 is imbalanced)
counts = np.bincount([eval_ds._row_to_record(eval_ds.df.iloc[i])['diagnosis'] for i in tr], minlength=2)
cw = torch.tensor(counts.sum() / (counts + 1e-6), dtype=torch.float32)
cw = cw / cw.mean()
print('class weights:', cw.tolist())

class weights: [0.3836745023727417, 1.6163256168365479]


In [4]:
# Train each architecture sequentially
ckpt_dir = PROJECT_ROOT / cfg['paths']['checkpoints_dir']
histories = {}
for arch in cfg['baseline']['architectures']:
    print(f'\n==== Training {arch} ====')
    model = build_baseline(arch, pretrained=cfg['baseline']['pretrained'])
    out = train_baseline(
        model, train_loader, val_loader,
        epochs=cfg['baseline']['epochs'],
        learning_rate=cfg['baseline']['learning_rate'],
        weight_decay=cfg['baseline']['weight_decay'],
        early_stopping_patience=cfg['baseline']['early_stopping_patience'],
        device=device,
        checkpoint_path=ckpt_dir / f'{arch}.pt',
        class_weights=cw,
    )
    histories[arch] = out
    test_loss, test_auroc = evaluate_baseline(model, test_loader, device=device)
    print(f'{arch} TEST AUROC={test_auroc:.4f}')


==== Training efficientnet_b0 ====


epoch 20/20: 100%|██████████| 501/501 [01:41<00:00,  4.95it/s]


efficientnet_b0 TEST AUROC=0.9716

==== Training resnet50 ====
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /Users/deepesh/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 86.5MB/s]
epoch 20/20: 100%|██████████| 501/501 [02:09<00:00,  3.88it/s]
INFO:src.models.baseline_cnn:Early stopping at epoch 19


resnet50 TEST AUROC=0.9676
